# EXA-STAR: finalize an evolved ViT-MAE into a competitive model (Kaggle T4x2)

The evolutionary search only trains each genome *briefly*, so its "best" genome is a lightly-trained
**topology** (and, given a noisy proxy fitness, not necessarily even the best topology). This
notebook turns that into a finished model, in two stages:

1. **Re-rank** (checkpoint mode): re-train each of the top-K survivors at a higher budget and
   re-score on the validation split, so the winner is chosen by a less-noisy signal.
2. **Full training**: freeze the winning topology and train it for a long dedicated run
   (warmup+cosine LR, periodic validation, early stopping, best-validation checkpoint), then report
   the honest **held-out TEST R^2** vs BrainLM.

It imports the logic from `evaluation_scripts/train_final_model.py` (one source of truth), so this
notebook stays a thin, editable driver -- change a value in the config cell and re-run.

> **Reuse the SAME split + stats the evolution run used.** Point `SPLIT_PATH` / `STATS_PATH` at the
> exact files your search wrote, or the "held-out test" leaks subjects the model trained on.

## 1. Configuration

In [ ]:
import os

# --- repo + data locations (EDIT these; match your evolution notebook) ---
REPO_PATH = "/kaggle/working/exa-star"                 # where the repo is cloned on Kaggle
REPO_URL = "https://github.com/axj2613/exa-ae.git"     # your fork; add a token here if private
REPO_BRANCH = "autoencoder-aryan"
HCP_ROOT = "/kaggle/working/hcp_complete"              # dir containing the <subject_id>/ subdirs
ATLAS_COORDS = os.path.join(REPO_PATH, "datasets/hcp/atlases/A424_Coordinates.dat")

WORKDIR = "/kaggle/working"
# these MUST be the same files the evolution run produced, so the test split is truly held out:
SPLIT_PATH = os.path.join(WORKDIR, "subject_split.json")
STATS_PATH = os.path.join(WORKDIR, "norm_stats.npz")
LENGTH_INDEX_PATH = os.path.join(WORKDIR, "length_index.json")

# --- what to finalize ---
# "checkpoint": re-rank the checkpoint's whole population, then full-train the winner.
# "genome":     skip re-ranking; full-train one specific genome .pkl directly.
SOURCE = "checkpoint"
CHECKPOINT_PATH = os.path.join(WORKDIR, "evolution_checkpoint.pkl")
GENOME_PATH = os.path.join(WORKDIR, "best_genome.pkl")
OUTPUT_PATH = os.path.join(WORKDIR, "final_model.pkl")   # best-validation model is saved here

# --- full-training schedule (this is the run that actually competes on reconstruction) ---
TOTAL_STEPS = 1500          # raise if val MSE is still descending at the end
BATCH_SIZE = 128             # full-training batch (bigger than the search's tiny proxy batch)
VAL_EVERY = 500              # validate + maybe-checkpoint every N steps
VAL_BATCHES = 32             # validation batches averaged per check
TEST_BATCHES = 64            # batches for the final held-out test R^2
PATIENCE = 8                 # early stop after this many checks without val improvement
LR = 6.73e-05
MIN_LR = 1e-5               # cosine floor
WARMUP_STEPS = 300
DROPOUT = None               # e.g. 0.2 to raise dropout for the long run; None keeps as-evolved

# --- re-rank budget (SOURCE == "checkpoint" only) ---
RERANK_ITERS = 6
RERANK_BPI = 100
RERANK_BATCH_SIZE = 16

## 2. Fetch the repo + imports

In [9]:
import subprocess
import sys

if not os.path.isdir(REPO_PATH):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull", "origin", REPO_BRANCH], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

import pickle
from types import SimpleNamespace
# expandable segments reduce caching-allocator fragmentation (must be set before torch inits CUDA).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch

from evolution.checkpoint import load_checkpoint
from time_series.hcp_window_dataset import HCPWindowDataset
from evaluation_scripts.train_final_model import rerank, full_train, evaluate, set_dropout

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

Already up to date.
device: cuda Tesla T4


From https://github.com/axj2613/exa-ae
 * branch            autoencoder-aryan -> FETCH_HEAD


## 3. Load the winner / candidates + build the dataset

In [10]:
# bundle the config into the `args` object the train_final_model functions expect
args = SimpleNamespace(
    output=OUTPUT_PATH, lr=LR, min_lr=MIN_LR, warmup_steps=WARMUP_STEPS,
    total_steps=TOTAL_STEPS, batch_size=BATCH_SIZE, val_every=VAL_EVERY,
    val_batches=VAL_BATCHES, test_batches=TEST_BATCHES, patience=PATIENCE,
    rerank_iters=RERANK_ITERS, rerank_bpi=RERANK_BPI, rerank_batch_size=RERANK_BATCH_SIZE,
)

# load the genome/population first to learn the window length, then build the matching dataset
candidates = None
if SOURCE == "checkpoint":
    state = load_checkpoint(CHECKPOINT_PATH)
    candidates = list(state["population_strategy"].population)
    window_length = candidates[0].window_length
    print(f"loaded {len(candidates)} survivors from {CHECKPOINT_PATH}")
elif SOURCE == "genome":
    with open(GENOME_PATH, "rb") as genome_file:
        winner = pickle.load(genome_file)
    window_length = winner.window_length
    print(f"loaded a single genome from {GENOME_PATH}")
else:
    raise ValueError(f"SOURCE must be 'checkpoint' or 'genome', got {SOURCE!r}")

dataset = HCPWindowDataset(
    root_dir=HCP_ROOT, atlas_coordinates_filename=ATLAS_COORDS,
    window_length=window_length, split_path=SPLIT_PATH,
    stats_path=STATS_PATH, length_index_path=LENGTH_INDEX_PATH,
)
print("window_length:", window_length, "| num_parcels:", dataset.num_parcels)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/final_model/evolution_checkpoint.pkl'

## 4. Re-rank the survivors (checkpoint mode only)

Re-trains each survivor at the higher re-rank budget and re-scores on validation, then picks the
winner by that less-noisy signal. Skipped in `genome` mode.

In [ ]:
if candidates is not None:
    winner = rerank(candidates, dataset, device, args)
    print(f"\nselected winner: gen {winner.generation_number}, "
          f"{winner.complexity['total_active_parameters']:,} params")

if DROPOUT is not None:
    set_dropout(winner, DROPOUT)
    print(f"dropout overridden to {DROPOUT} for the final run")


=== re-ranking 16 survivors at 6x100 steps ===
iteration 0 train loss: 0.803961
iteration 1 train loss: 0.815962
iteration 2 train loss: 0.811302
iteration 3 train loss: 0.841120
iteration 4 train loss: 0.809411
iteration 5 train loss: 0.789386
final fitness (validation MSE): 0.792984 | active params: 608,348 (594,824 evolved) | active hidden nodes: 3, edges: 8 | types: {'AttentionBlockNode': 3} (encoder: {'AttentionBlockNode': 2}, decoder: {'AttentionBlockNode': 1})
iteration 0 train loss: 0.801178
iteration 1 train loss: 0.817313
iteration 2 train loss: 0.781106
iteration 3 train loss: 0.771020
iteration 4 train loss: 0.796474
iteration 5 train loss: 0.810772
final fitness (validation MSE): 0.790993 | active params: 608,345 (594,821 evolved) | active hidden nodes: 3, edges: 5 | types: {'AttentionBlockNode': 3} (encoder: {'AttentionBlockNode': 1}, decoder: {'AttentionBlockNode': 2})
iteration 0 train loss: 0.807707
iteration 1 train loss: 0.775971
iteration 2 train loss: 0.788930
ite

## 5. Full training

The long, dedicated run: warmup+cosine LR, periodic validation, early stopping, best-validation
checkpoint saved to `OUTPUT_PATH`. Watch the `val MSE` / `val R2` prints -- if MSE is still
descending at the last step, raise `TOTAL_STEPS`; if it's glued near ~1.0 by step 2-3k, stop and
revisit capacity / LR / mask ratio.

In [ ]:
full_train(winner, dataset, device, args)


=== full training: 10000 steps, batch 128, val every 500, patience 8 ===
step    500  lr 9.99e-04  train 0.77944  val MSE 0.78795  val R2 0.2093  <- best
step   1000  lr 9.87e-04  train 0.77854  val MSE 0.78534  val R2 0.2138  <- best
step   1500  lr 9.63e-04  train 0.77245  val MSE 0.77877  val R2 0.2167  <- best
step   2000  lr 9.27e-04  train 0.78962  val MSE 0.77448  val R2 0.2180  <- best
step   2500  lr 8.80e-04  train 0.79385  val MSE 0.77277  val R2 0.2224  <- best
step   3000  lr 8.22e-04  train 0.78756  val MSE 0.77395  val R2 0.2240
step   3500  lr 7.57e-04  train 0.77448  val MSE 0.77385  val R2 0.2216
step   4000  lr 6.85e-04  train 0.78081  val MSE 0.76629  val R2 0.2269  <- best
step   4500  lr 6.08e-04  train 0.77812  val MSE 0.76913  val R2 0.2270
step   5000  lr 5.29e-04  train 0.76885  val MSE 0.76073  val R2 0.2353  <- best
step   5500  lr 4.49e-04  train 0.76447  val MSE 0.75448  val R2 0.2394  <- best
step   6000  lr 3.70e-04  train 0.75318  val MSE 0.75433  val 

: 

## 6. Held-out TEST R^2 (the honest number)

Reconstruction R^2 on the TEST split -- untouched during training and selection -- computed once on
the best-validation model, for comparison against BrainLM (~0.46 UKB / ~0.28 HCP).

In [ ]:
with open(OUTPUT_PATH, "rb") as best_file:
    best = pickle.load(best_file)
best.to(device)
test_mse, test_r2 = evaluate(best, dataset, device, "test", BATCH_SIZE, TEST_BATCHES, device.type == "cuda")
print(f"=== HELD-OUT TEST (best-val model, {TEST_BATCHES} batches) ===")
print(f"reconstruction R^2 = {test_r2:.4f}   (MSE {test_mse:.5f})")
print("BrainLM reference: masked-reconstruction R^2 ~0.46 (UKB) / ~0.28 (HCP)")

=== HELD-OUT TEST (best-val model, 64 batches) ===
reconstruction R^2 = 0.2434   (MSE 0.75030)
BrainLM reference: masked-reconstruction R^2 ~0.46 (UKB) / ~0.28 (HCP)


In [ ]:
from IPython.display import FileLink, display
display(FileLink(OUTPUT_PATH))   # click to download the finished model

/kaggle/working/final_model/final_model.pkl

## Notes

- **Same split/stats as evolution.** This notebook reuses `SPLIT_PATH` / `STATS_PATH`; if those
  files aren't the ones the search wrote, the "held-out" test isn't held out. Keep them on Kaggle
  persistence (Settings -> Persistence: Files only) so they survive across sessions.
- **Editing the logic.** The heavy lifting lives in `evaluation_scripts/train_final_model.py`
  (`rerank`, `full_train`, `evaluate`); this notebook only wires config -> those functions. Change
  the schedule here; change the algorithm there and `git pull` (cell 2 pulls on every run).
- **Re-rank budget vs. full budget.** Re-rank (`RERANK_ITERS x RERANK_BPI`) is intentionally cheap --
  its job is to filter clear duds among the survivors, not to fully train them. The real learning
  happens in **full training**, so don't expect re-rank to dramatically reshuffle the leaderboard.
- **Single-GPU is fine here.** Unlike the search, finalization trains one model, so it uses one GPU.